In [0]:
%sql
create external location IF not exists dea_course_ext_dl_spr_ridesshare
URL 'abfss://ridesproject@deacourseextdlspr.dfs.core.windows.net/'
WITH (storage credential dea_course_ext_sc)

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS rides_catalog
MANAGED LOCATION 'abfss://ridesproject@deacourseextdlspr.dfs.core.windows.net/'
COMMENT "This is the catalog for the rides Data Lakehouse"

In [0]:
%sql
create schema if not exists rides_catalog.bronze;


In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://ridesproject@deacourseextdlspr.dfs.core.windows.net/landing/rides_data.csv")

In [0]:
from pyspark.sql.functions import current_timestamp

bronze_df = df.withColumn("ingestion_time", current_timestamp())


In [0]:
bronze_df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true").saveAsTable("rides_catalog.bronze.rides_raw")

In [0]:
%sql
select * from rides_catalog.bronze.rides_raw;